# Final supervised encoder experiments

Unified final pipeline for BERT, RoBERTa, DistilBERT, and ModernBERT.

All four models use the same:
- 717-passage dataset and book-disjoint train/DEV/TEST split
- label mapping and 512-token input limit
- 12-configuration hyperparameter grid
- DEV macro-F1 model selection
- selected-model TEST evaluation
- 1,000-replicate book-level cluster bootstrap

This notebook is configured to run all four models from scratch under the same executable pipeline.
Temporary checkpoints remain on Colab local disk; Google Drive stores only persistent results and final best models.

## Resume behavior

This version is interruption-safe:
- completed DEV configurations are detected from their saved Drive JSONs and skipped;
- an interrupted in-progress configuration is rerun because it has no completed DEV JSON;
- if Colab disconnects after DEV metrics were saved but before the selected best model weights were persisted, only that one selected best configuration is retrained to regenerate the weights;
- completed TEST results are reused when both the TEST JSON and prediction CSV exist.

Temporary Trainer checkpoints still remain on Colab local disk only.


## Colab setup

1. Select a GPU runtime.
2. Mount Google Drive.
3. Run the package-install cell once in a fresh runtime.
4. Verify `DATA_PATH` and `OUTPUT_ROOT` in the configuration cell.
5. Run cells top-to-bottom.

The full benchmark is **48 training runs** (4 models × 12 configurations).
Temporary Hugging Face checkpoints and changing best-model candidates are kept under `/content` so they do not fill Google Drive or Google Drive Trash.


In [ ]:
%pip install -q transformers==4.48.3 tokenizers accelerate pandas numpy scipy scikit-learn

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


## Configuration

In [ ]:
from pathlib import Path

# Final four-model encoder benchmark.
MODELS_TO_RUN = [
    "bert-base-uncased",
    "roberta-base",
    "distilbert-base-uncased",
    "answerdotai/ModernBERT-base",
]

SUPPORTED_MODELS = MODELS_TO_RUN.copy()

# Colab / Google Drive paths.
# Change these two paths only if your Drive layout is different.
DATA_PATH = Path(
    "/content/drive/MyDrive/Interiority_26Spring/Data/df_annotation_final.csv"
)
OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/Interiority_26Spring/encoder_model_results"
)

# IMPORTANT: all temporary Trainer artifacts stay on Colab local disk.
# This prevents epoch checkpoints / replaced candidate models from filling
# Google Drive or its Trash.
CHECKPOINT_ROOT = Path("/content/encoder_model_checkpoints")
CANDIDATE_ROOT = Path("/content/encoder_best_candidates")
EVAL_TMP_ROOT = Path("/content/encoder_test_eval")

LEARNING_RATES = [1e-5, 2e-5, 3e-5]
BATCH_SIZES = [16, 32]
WARMUP_RATIOS = [0.06, 0.1]
SEED = 42
MAX_LENGTH = 512
NUM_EPOCHS = 6
WEIGHT_DECAY = 0.01
EARLY_STOPPING_PATIENCE = 2
N_BOOTSTRAP = 1000
BOOTSTRAP_SEED = 42

# Resume behavior:
# - Completed DEV configurations with valid saved JSONs are skipped.
# - If Colab disconnects before a model family's final best weights are copied
#   to Drive, only that family's selected best configuration is retrained once
#   to regenerate the weights; completed grid configurations are not repeated.
# - Completed TEST results are also reused when their saved JSON and prediction
#   CSV are both present.
RESUME_COMPLETED_CONFIGS = True
RESUME_TEST_RESULTS = True

unknown_models = sorted(set(MODELS_TO_RUN).difference(SUPPORTED_MODELS))
assert not unknown_models, f"Unsupported model checkpoint(s): {unknown_models}"
assert len(MODELS_TO_RUN) == len(set(MODELS_TO_RUN)), "Duplicate model entries"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
CANDIDATE_ROOT.mkdir(parents=True, exist_ok=True)
EVAL_TMP_ROOT.mkdir(parents=True, exist_ok=True)

CONFIGS_PER_MODEL = (
    len(LEARNING_RATES) * len(BATCH_SIZES) * len(WARMUP_RATIOS)
)
TOTAL_RUNS = len(MODELS_TO_RUN) * CONFIGS_PER_MODEL

print("Models to run:", MODELS_TO_RUN)
print("Configurations per model:", CONFIGS_PER_MODEL)
print("Total training runs:", TOTAL_RUNS)
print("Drive output root:", OUTPUT_ROOT)
print("Temporary checkpoint root:", CHECKPOINT_ROOT)


Models to run: ['bert-base-uncased', 'roberta-base', 'distilbert-base-uncased', 'answerdotai/ModernBERT-base']
Configurations per model: 12
Total training runs: 48
Drive output root: /content/drive/MyDrive/Interiority_26Spring/encoder_model_results
Temporary checkpoint root: /content/encoder_model_checkpoints


## Imports and runtime check

In [ ]:
import itertools
import json
import shutil
import warnings

import numpy as np
import pandas as pd
import torch
import transformers
from scipy.stats import spearmanr
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

assert transformers.__version__ == "4.48.3", (
    f"Expected transformers==4.48.3; found {transformers.__version__}"
)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No CUDA GPU detected. Use a Colab GPU runtime for training.")

Transformers: 4.48.3
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


# Data checks and split setup

In [ ]:
df = pd.read_csv(DATA_PATH)

required_columns = {"title", "paragraph", "split", "gold_label", "label_id"}
missing_columns = required_columns.difference(df.columns)
assert not missing_columns, f"Missing required columns: {sorted(missing_columns)}"

# Preserve the source notebook's non-model paragraph-length repair.
if "paragraph_length" in df.columns:
    mask = df["paragraph_length"].isna()
    df.loc[mask, "paragraph_length"] = df.loc[mask, "paragraph"].apply(
        lambda text: len(str(text).split())
    )

assert len(df) == 717, f"Expected 717 passages; found {len(df)}"
assert set(df["split"].unique()) == {"train", "dev", "test"}

train_df = df[df["split"] == "train"].reset_index(drop=True)
dev_df = df[df["split"] == "dev"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

expected_split_sizes = {"train": 357, "dev": 120, "test": 240}
actual_split_sizes = {
    "train": len(train_df),
    "dev": len(dev_df),
    "test": len(test_df),
}
assert actual_split_sizes == expected_split_sizes, actual_split_sizes

expected_label_mapping = {"none": 0, "low": 1, "high": 2}
actual_label_mapping = (
    df[["gold_label", "label_id"]]
    .drop_duplicates()
    .set_index("gold_label")["label_id"]
    .astype(int)
    .to_dict()
)
assert actual_label_mapping == expected_label_mapping, actual_label_mapping

split_books = {
    "train": set(train_df["title"]),
    "dev": set(dev_df["title"]),
    "test": set(test_df["title"]),
}
assert split_books["train"].isdisjoint(split_books["dev"])
assert split_books["train"].isdisjoint(split_books["test"])
assert split_books["dev"].isdisjoint(split_books["test"])
assert len(split_books["test"]) == 6, (
    f"Expected 6 TEST books; found {len(split_books['test'])}"
)

LABEL_NAMES = ["none", "low", "high"]
NUM_LABELS = len(LABEL_NAMES)

print("Split sizes:", actual_split_sizes)
print("Books per split:", {name: len(books) for name, books in split_books.items()})
print("Label mapping:", actual_label_mapping)
print(pd.crosstab(df["split"], df["gold_label"]))

Split sizes: {'train': 357, 'dev': 120, 'test': 240}
Books per split: {'train': 9, 'dev': 3, 'test': 6}
Label mapping: {'high': 2, 'low': 1, 'none': 0}
gold_label  high  low  none
split                      
dev           53   50    17
test         104   93    43
train        170  131    56


# Dataset class

In [ ]:
class InteriorityDataset(Dataset):
    """Tokenize passages on demand in the HuggingFace Trainer format."""

    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

# Metrics and book-level cluster bootstrap

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(
            labels, predictions, average="macro", zero_division=0
        ),
        "weighted_f1": f1_score(
            labels, predictions, average="weighted", zero_division=0
        ),
        "micro_f1": f1_score(
            labels, predictions, average="micro", zero_division=0
        ),
    }


def spearman_metric(labels, predictions):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return spearmanr(labels, predictions).statistic


def book_bootstrap_ci(
    labels,
    predictions,
    titles,
    metric_fn,
    n_bootstrap=1000,
    confidence=0.95,
    seed=42,
):
    """Percentile CI from paired resampling of complete book clusters."""
    labels = np.asarray(labels)
    predictions = np.asarray(predictions)
    titles = np.asarray(titles)
    assert len(labels) == len(predictions) == len(titles)

    books = pd.unique(titles)
    indices_by_book = {
        book: np.flatnonzero(titles == book) for book in books
    }
    rng = np.random.default_rng(seed)
    bootstrap_scores = []

    for _ in range(n_bootstrap):
        sampled_books = rng.choice(books, size=len(books), replace=True)
        sampled_indices = np.concatenate(
            [indices_by_book[book] for book in sampled_books]
        )
        score = metric_fn(
            labels[sampled_indices], predictions[sampled_indices]
        )
        bootstrap_scores.append(
            float(score) if np.isfinite(score) else np.nan
        )

    valid_scores = np.asarray(bootstrap_scores, dtype=float)
    valid_scores = valid_scores[np.isfinite(valid_scores)]
    if len(valid_scores) == 0:
        lower = upper = np.nan
    else:
        alpha = (1 - confidence) / 2
        lower, upper = np.percentile(
            valid_scores, [100 * alpha, 100 * (1 - alpha)]
        )

    point = metric_fn(labels, predictions)
    return {
        "point": float(point),
        "lower": float(lower),
        "upper": float(upper),
        "valid_replicates": int(len(valid_scores)),
        "total_replicates": int(n_bootstrap),
    }

In [ ]:
def evaluate_model(
    trainer,
    dataset,
    df_reference,
    split_name,
    bootstrap_by_book=False,
    n_bootstrap=1000,
):
    """Evaluate a split; apply book-cluster CIs only when requested."""
    output = trainer.predict(dataset)
    logits = output.predictions
    labels = output.label_ids
    predictions = np.argmax(logits, axis=-1)
    shifted_logits = logits - logits.max(axis=1, keepdims=True)
    probabilities = np.exp(shifted_logits)
    probabilities /= probabilities.sum(axis=1, keepdims=True)
    titles = df_reference.reset_index(drop=True)["title"].to_numpy()

    metric_functions = {
        "accuracy": lambda y, p: accuracy_score(y, p),
        "macro_f1": lambda y, p: f1_score(
            y, p, average="macro", zero_division=0
        ),
        "weighted_f1": lambda y, p: f1_score(
            y, p, average="weighted", zero_division=0
        ),
        "micro_f1": lambda y, p: f1_score(
            y, p, average="micro", zero_division=0
        ),
        "spearman": spearman_metric,
    }

    print(f"\n{'=' * 60}\n{split_name} EVALUATION\n{'=' * 60}")
    results = {}
    for metric_name, metric_fn in metric_functions.items():
        point = float(metric_fn(labels, predictions))
        results[metric_name] = point
        if bootstrap_by_book:
            interval = book_bootstrap_ci(
                labels,
                predictions,
                titles,
                metric_fn,
                n_bootstrap=n_bootstrap,
                seed=BOOTSTRAP_SEED,
            )
            results[f"{metric_name}_lo"] = interval["lower"]
            results[f"{metric_name}_hi"] = interval["upper"]
            results[f"{metric_name}_bootstrap_valid"] = (
                interval["valid_replicates"]
            )
            results[f"{metric_name}_bootstrap_total"] = (
                interval["total_replicates"]
            )
            print(
                f"{metric_name:<15} {point:.4f} "
                f"[{interval['lower']:.4f}, {interval['upper']:.4f}] "
                f"valid={interval['valid_replicates']}/{n_bootstrap}"
            )
        else:
            print(f"{metric_name:<15} {point:.4f}")

    class_report = classification_report(
        labels,
        predictions,
        labels=[0, 1, 2],
        target_names=LABEL_NAMES,
        digits=4,
        zero_division=0,
        output_dict=True,
    )
    results["classification_report"] = class_report
    print("\nPer-class precision / recall / F1:")
    print(pd.DataFrame(class_report).T.to_string())

    for class_id, label_name in enumerate(LABEL_NAMES):
        class_f1 = lambda y, p, class_id=class_id: f1_score(
            y,
            p,
            average=None,
            labels=[0, 1, 2],
            zero_division=0,
        )[class_id]
        results[f"f1_{label_name}"] = float(class_f1(labels, predictions))
        if bootstrap_by_book:
            interval = book_bootstrap_ci(
                labels,
                predictions,
                titles,
                class_f1,
                n_bootstrap=n_bootstrap,
                seed=BOOTSTRAP_SEED,
            )
            results[f"f1_{label_name}_lo"] = interval["lower"]
            results[f"f1_{label_name}_hi"] = interval["upper"]
            results[f"f1_{label_name}_bootstrap_valid"] = (
                interval["valid_replicates"]
            )
            results[f"f1_{label_name}_bootstrap_total"] = (
                interval["total_replicates"]
            )

    matrix = confusion_matrix(labels, predictions, labels=[0, 1, 2])
    results["confusion_matrix"] = matrix
    print("\nConfusion matrix (rows=true, columns=predicted):")
    print(pd.DataFrame(matrix, index=LABEL_NAMES, columns=LABEL_NAMES))

    evaluation_frame = df_reference.reset_index(drop=True).copy()
    evaluation_frame["predicted_label_id"] = predictions
    per_book = {}
    print("\nPer-book macro F1:")
    for book, group in evaluation_frame.groupby("title"):
        score = f1_score(
            group["label_id"],
            group["predicted_label_id"],
            average="macro",
            zero_division=0,
        )
        per_book[book] = float(score)
        print(f"{book:<45} n={len(group):>3} F1={score:.4f}")

    results["per_book"] = per_book
    results["predictions"] = predictions
    results["labels"] = labels
    results["logits"] = logits
    results["probabilities"] = probabilities
    return results

# Training function

In [ ]:
def model_slug(model_name):
    return model_name.replace("/", "_")


def config_tag(model_name, learning_rate, batch_size, warmup_ratio):
    return (
        f"{model_slug(model_name)}_lr{learning_rate}_"
        f"b{batch_size}_warmup{warmup_ratio}"
    )


def clear_device_cache():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def train_one_model(
    model_name,
    train_frame,
    dev_frame,
    seed=42,
    max_length=512,
    num_epochs=6,
    batch_size=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
):
    """Fine-tune one configuration and return DEV point estimates."""
    print(
        f"\n{'#' * 60}\nTraining: {model_name}\n"
        f"lr={learning_rate} batch={batch_size} warmup={warmup_ratio} seed={seed}\n"
        f"{'#' * 60}"
    )
    set_seed(seed)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=NUM_LABELS,
        id2label={0: "none", 1: "low", 2: "high"},
        label2id={"none": 0, "low": 1, "high": 2},
    )

    train_dataset = InteriorityDataset(
        train_frame["paragraph"],
        train_frame["label_id"],
        tokenizer,
        max_length,
    )
    dev_dataset = InteriorityDataset(
        dev_frame["paragraph"],
        dev_frame["label_id"],
        tokenizer,
        max_length,
    )

    run_tag = config_tag(
        model_name, learning_rate, batch_size, warmup_ratio
    )

    # Keep epoch checkpoints on Colab local disk, never Google Drive.
    checkpoint_dir = CHECKPOINT_ROOT / run_tag
    if checkpoint_dir.exists():
        shutil.rmtree(checkpoint_dir)

    training_args = TrainingArguments(
        output_dir=str(checkpoint_dir),
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        learning_rate=learning_rate,
        warmup_ratio=warmup_ratio,
        weight_decay=WEIGHT_DECAY,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=1,
        seed=seed,
        report_to="none",
        fp16=torch.cuda.is_available(),
        disable_tqdm=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=dev_dataset,
        compute_metrics=compute_metrics,
        # Early stopping and best-checkpoint selection use DEV macro F1.
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=EARLY_STOPPING_PATIENCE
            )
        ],
    )

    try:
        trainer.train()
        dev_results = evaluate_model(
            trainer,
            dev_dataset,
            dev_frame,
            "DEV",
            bootstrap_by_book=False,
        )
        return trainer, dev_results
    finally:
        # Trainer has already reloaded its best checkpoint into memory.
        # Remove local epoch checkpoints after each configuration.
        if checkpoint_dir.exists():
            shutil.rmtree(checkpoint_dir)


# Result-saving helpers

In [ ]:
def make_json_safe(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {str(key): make_json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_safe(item) for item in value]
    return value


def save_result_json(results, model_name, config, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "model": model_name,
        "config": config,
        "results": make_json_safe(results),
    }
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"Saved → {path}")


def save_test_predictions(test_frame, results, path):
    prediction_frame = test_frame[["title", "gold_label", "label_id"]].copy()
    prediction_frame["predicted_label_id"] = results["predictions"].astype(int)
    prediction_frame["predicted_label"] = prediction_frame["predicted_label_id"].map(
        dict(enumerate(LABEL_NAMES))
    )
    for class_id, label_name in enumerate(LABEL_NAMES):
        prediction_frame[f"logit_{label_name}"] = results["logits"][:, class_id]
        prediction_frame[f"probability_{label_name}"] = (
            results["probabilities"][:, class_id]
        )
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    prediction_frame.to_csv(path, index=False)
    print(f"Saved → {path}")

def configs_match(saved_config, expected_config):
    """Compare the three grid fields in a saved result against the expected config."""
    keys = ["learning_rate", "batch_size", "warmup_ratio"]
    try:
        return all(
            float(saved_config[key]) == float(expected_config[key])
            for key in keys
        )
    except (KeyError, TypeError, ValueError):
        return False


def load_saved_result_json(path, expected_model=None, expected_config=None):
    """Load a saved result only if it is complete and matches this run."""
    path = Path(path)
    if not path.exists():
        return None

    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"Saved result is unreadable; will rerun: {path} ({exc})")
        return None

    if expected_model is not None and payload.get("model") != expected_model:
        print(f"Saved result model mismatch; will rerun: {path}")
        return None

    saved_config = payload.get("config", {})
    if expected_config is not None and not configs_match(
        saved_config, expected_config
    ):
        print(f"Saved result config mismatch; will rerun: {path}")
        return None

    results = payload.get("results")
    if not isinstance(results, dict) or "macro_f1" not in results:
        print(f"Saved result is incomplete; will rerun: {path}")
        return None

    return results


def model_dir_has_weights(path):
    """Return True for normal or sharded Hugging Face weight layouts."""
    path = Path(path)
    if not path.exists():
        return False
    candidates = [
        "model.safetensors",
        "pytorch_model.bin",
        "model.safetensors.index.json",
        "pytorch_model.bin.index.json",
    ]
    return any((path / name).exists() for name in candidates)


def saved_best_model_is_valid(model_dir, expected_config):
    """Verify that a persisted best-model directory is loadable enough to reuse."""
    model_dir = Path(model_dir)
    config_path = model_dir / "best_config.json"
    if not model_dir_has_weights(model_dir) or not config_path.exists():
        return False
    try:
        saved_config = json.loads(config_path.read_text(encoding="utf-8"))
    except Exception:
        return False
    return configs_match(saved_config, expected_config)



# DEV grid search and best-model selection

Each selected model runs the same 12 configurations. Only DEV macro F1 determines the selected configuration. TEST is not loaded by the training function and is not evaluated in this section.

In [ ]:
grid = list(itertools.product(LEARNING_RATES, BATCH_SIZES, WARMUP_RATIOS))
best_per_model = {}

for model_name in MODELS_TO_RUN:
    print(f"\n{'=' * 70}\nGRID / RESUME CHECK: {model_name}\n{'=' * 70}")

    best_dev_f1 = -np.inf
    best_config = None
    best_dev_results = None

    # The changing current-best candidate is local only.
    candidate_dir = CANDIDATE_ROOT / model_slug(model_name)
    if candidate_dir.exists():
        shutil.rmtree(candidate_dir)
    candidate_tag = None

    completed_count = 0
    trained_count = 0

    for learning_rate, batch_size, warmup_ratio in grid:
        config = {
            "learning_rate": learning_rate,
            "batch_size": batch_size,
            "warmup_ratio": warmup_ratio,
        }
        run_tag = config_tag(
            model_name, learning_rate, batch_size, warmup_ratio
        )
        dev_path = OUTPUT_ROOT / "dev" / f"{run_tag}_dev_results.json"

        dev_results = None
        trainer = None

        if RESUME_COMPLETED_CONFIGS:
            dev_results = load_saved_result_json(
                dev_path,
                expected_model=model_name,
                expected_config=config,
            )

        if dev_results is not None:
            completed_count += 1
            print(
                f"SKIP completed DEV config: {run_tag} "
                f"(macro F1={float(dev_results['macro_f1']):.4f})"
            )
        else:
            trainer, dev_results = train_one_model(
                model_name=model_name,
                train_frame=train_df,
                dev_frame=dev_df,
                seed=SEED,
                max_length=MAX_LENGTH,
                num_epochs=NUM_EPOCHS,
                batch_size=batch_size,
                learning_rate=learning_rate,
                warmup_ratio=warmup_ratio,
            )
            trained_count += 1

            # Small DEV JSONs are safe to save directly to Drive.
            save_result_json(
                dev_results, model_name, config, dev_path
            )

        dev_macro_f1 = float(dev_results["macro_f1"])

        if dev_macro_f1 > best_dev_f1:
            best_dev_f1 = dev_macro_f1
            best_config = config.copy()
            best_dev_results = dev_results
            best_tag_so_far = run_tag

            # We only have weights in memory if this config trained in the
            # current runtime. Save that current-best candidate locally.
            if trainer is not None:
                if candidate_dir.exists():
                    shutil.rmtree(candidate_dir)
                trainer.model.save_pretrained(candidate_dir)
                AutoTokenizer.from_pretrained(model_name).save_pretrained(
                    candidate_dir
                )
                candidate_tag = run_tag

        if trainer is not None:
            del trainer
            clear_device_cache()

    if best_config is None:
        raise RuntimeError(f"No valid DEV results found for {model_name}")

    best_tag = config_tag(
        model_name,
        best_config["learning_rate"],
        best_config["batch_size"],
        best_config["warmup_ratio"],
    )
    best_model_dir = OUTPUT_ROOT / "best_models" / best_tag

    # If this model family completed in a previous session and its final best
    # weights are already safely on Drive, reuse them.
    if RESUME_COMPLETED_CONFIGS and saved_best_model_is_valid(
        best_model_dir, best_config
    ):
        print(f"REUSE persisted best model → {best_model_dir}")

    else:
        # If the selected best config was trained in this runtime, its weights
        # are already in candidate_dir. Otherwise Colab likely disconnected
        # after saving DEV metrics but before persisting the selected weights.
        # In that case, retrain ONLY the selected best configuration once.
        if candidate_tag != best_tag or not model_dir_has_weights(candidate_dir):
            print(
                "Selected best DEV configuration has saved metrics but no "
                "recoverable local weights. Retraining ONLY this one best "
                "configuration to regenerate weights:"
            )
            print(best_config)

            trainer, regenerated_dev_results = train_one_model(
                model_name=model_name,
                train_frame=train_df,
                dev_frame=dev_df,
                seed=SEED,
                max_length=MAX_LENGTH,
                num_epochs=NUM_EPOCHS,
                batch_size=best_config["batch_size"],
                learning_rate=best_config["learning_rate"],
                warmup_ratio=best_config["warmup_ratio"],
            )

            if candidate_dir.exists():
                shutil.rmtree(candidate_dir)
            trainer.model.save_pretrained(candidate_dir)
            AutoTokenizer.from_pretrained(model_name).save_pretrained(
                candidate_dir
            )
            candidate_tag = best_tag

            print(
                "Regenerated best-config DEV macro F1:",
                f"{float(regenerated_dev_results['macro_f1']):.4f}",
                "| originally saved:",
                f"{best_dev_f1:.4f}",
            )

            del trainer
            clear_device_cache()

        # Persist ONE final selected model to Drive.
        best_model_dir.parent.mkdir(parents=True, exist_ok=True)
        if best_model_dir.exists():
            shutil.rmtree(best_model_dir)
        shutil.copytree(candidate_dir, best_model_dir)
        (best_model_dir / "best_config.json").write_text(
            json.dumps(best_config, indent=2), encoding="utf-8"
        )
        print(f"Persisted final best model → {best_model_dir}")

    if candidate_dir.exists():
        shutil.rmtree(candidate_dir)

    best_per_model[model_name] = {
        "config": best_config,
        "dev_results": best_dev_results,
        "model_dir": best_model_dir,
        "tag": best_tag,
    }

    print(
        f"Best {model_name} config: {best_config}; "
        f"DEV macro F1={best_dev_f1:.4f}"
    )
    print(
        f"Resume summary: {completed_count} configs reused, "
        f"{trained_count} configs newly trained."
    )



GRID / RESUME CHECK: bert-base-uncased

############################################################
Training: bert-base-uncased
lr=1e-05 batch=16 warmup=0.06 seed=42
############################################################


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

... (115 outputs left out)


DEV EVALUATION
accuracy        0.6583
macro_f1        0.5777
weighted_f1     0.6443
micro_f1        0.6583
spearman        0.5141

Per-class precision / recall / F1:
              precision    recall  f1-score     support
none           0.666667  0.235294  0.347826   17.000000
low            0.590909  0.780000  0.672414   50.000000
high           0.750000  0.679245  0.712871   53.000000
accuracy       0.658333  0.658333  0.658333    0.658333
macro avg      0.669192  0.564846  0.577704  120.000000
weighted avg   0.671907  0.658333  0.644299  120.000000

Confusion matrix (rows=true, columns=predicted):
      none  low  high
none     4   11     2
low      1   39    10
high     1   16    36

Per-book macro F1:
Jane Eyre: An Autobiography                   n= 40 F1=0.4637
The Garden Party, and Other Stories           n= 40 F1=0.5799
The Great Gatsby                              n= 40 F1=0.6048
Saved → /content/drive/MyDrive/Interiority_26Spring/encoder_model_results/dev/distilbert-base-unc

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Micro F1
1,1.066100,0.989127,0.466667,0.246110,0.323318,0.466667
2,0.954400,0.924813,0.575000,0.391120,0.506506,0.575000
3,0.805100,0.869935,0.575000,0.414117,0.533625,0.575000
4,0.652600,0.850658,0.600000,0.492342,0.573322,0.600000
5,0.518800,0.824552,0.658333,0.577310,0.643774,0.658333
6,0.477600,0.836344,0.658333,0.577704,0.644299,0.658333



DEV EVALUATION
accuracy        0.6583
macro_f1        0.5777
weighted_f1     0.6443
micro_f1        0.6583
spearman        0.5141

Per-class precision / recall / F1:
              precision    recall  f1-score     support
none           0.666667  0.235294  0.347826   17.000000
low            0.590909  0.780000  0.672414   50.000000
high           0.750000  0.679245  0.712871   53.000000
accuracy       0.658333  0.658333  0.658333    0.658333
macro avg      0.669192  0.564846  0.577704  120.000000
weighted avg   0.671907  0.658333  0.644299  120.000000

Confusion matrix (rows=true, columns=predicted):
      none  low  high
none     4   11     2
low      1   39    10
high     1   16    36

Per-book macro F1:
Jane Eyre: An Autobiography                   n= 40 F1=0.4637
The Garden Party, and Other Stories           n= 40 F1=0.5799
The Great Gatsby                              n= 40 F1=0.6048
Saved → /content/drive/MyDrive/Interiority_26Spring/encoder_model_results/dev/distilbert-base-unc

# Selected-model TEST evaluation

Each DEV-selected model is evaluated once on the complete 240-passage TEST set. Point estimates use all 240 passages. Confidence intervals use 1,000 book-level cluster bootstrap replicates over the six TEST books.

In [ ]:
test_results = {}

for model_name, bundle in best_per_model.items():
    config = bundle["config"]
    run_tag = bundle["tag"]
    model_dir = bundle["model_dir"]

    test_json_path = OUTPUT_ROOT / "test" / f"{run_tag}_test_results.json"
    test_prediction_path = (
        OUTPUT_ROOT / "test" / f"{run_tag}_test_predictions.csv"
    )

    saved_test_results = None
    if (
        RESUME_TEST_RESULTS
        and test_json_path.exists()
        and test_prediction_path.exists()
    ):
        saved_test_results = load_saved_result_json(
            test_json_path,
            expected_model=model_name,
            expected_config=config,
        )

    if saved_test_results is not None:
        print(
            f"\nSKIP completed TEST evaluation: {model_name} "
            f"(macro F1={float(saved_test_results['macro_f1']):.4f})"
        )
        test_results[model_name] = saved_test_results
        continue

    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    test_dataset = InteriorityDataset(
        test_df["paragraph"],
        test_df["label_id"],
        tokenizer,
        MAX_LENGTH,
    )

    # TEST evaluation output is temporary; keep it on Colab local disk.
    eval_tmp_dir = EVAL_TMP_ROOT / run_tag
    if eval_tmp_dir.exists():
        shutil.rmtree(eval_tmp_dir)

    test_args = TrainingArguments(
        output_dir=str(eval_tmp_dir),
        per_device_eval_batch_size=config["batch_size"] * 2,
        seed=SEED,
        report_to="none",
    )
    trainer = Trainer(model=model, args=test_args)

    try:
        results = evaluate_model(
            trainer=trainer,
            dataset=test_dataset,
            df_reference=test_df,
            split_name="TEST",
            bootstrap_by_book=True,
            n_bootstrap=N_BOOTSTRAP,
        )
        test_results[model_name] = results

        save_result_json(
            results,
            model_name,
            config,
            test_json_path,
        )
        save_test_predictions(
            test_df,
            results,
            test_prediction_path,
        )
    finally:
        if eval_tmp_dir.exists():
            shutil.rmtree(eval_tmp_dir)
        del trainer, model, tokenizer
        clear_device_cache()



TEST EVALUATION
accuracy        0.6583 [0.5792, 0.7417] valid=1000/1000
macro_f1        0.5676 [0.4791, 0.6424] valid=1000/1000
weighted_f1     0.6374 [0.5563, 0.7203] valid=1000/1000
micro_f1        0.6583 [0.5792, 0.7417] valid=1000/1000
spearman        0.5730 [0.3946, 0.7061] valid=1000/1000

Per-class precision / recall / F1:
              precision    recall  f1-score     support
none           0.533333  0.186047  0.275862   43.000000
low            0.570312  0.784946  0.660633   93.000000
high           0.793814  0.740385  0.766169  104.000000
accuracy       0.658333  0.658333  0.658333    0.658333
macro avg      0.632487  0.570459  0.567555  240.000000
weighted avg   0.660538  0.658333  0.637427  240.000000

Confusion matrix (rows=true, columns=predicted):
      none  low  high
none     8   30     5
low      5   73    15
high     2   25    77

Per-book macro F1:
A farewell to arms                            n= 40 F1=0.5922
Dubliners                                     n= 40 F1=


TEST EVALUATION
accuracy        0.7125 [0.6667, 0.7751] valid=1000/1000
macro_f1        0.6488 [0.5934, 0.6990] valid=1000/1000
weighted_f1     0.7117 [0.6631, 0.7741] valid=1000/1000
micro_f1        0.7125 [0.6667, 0.7751] valid=1000/1000
spearman        0.7248 [0.6427, 0.7969] valid=1000/1000

Per-class precision / recall / F1:
              precision    recall  f1-score   support
none           0.454545  0.348837  0.394737   43.0000
low            0.615385  0.774194  0.685714   93.0000
high           0.933333  0.807692  0.865979  104.0000
accuracy       0.712500  0.712500  0.712500    0.7125
macro avg      0.667754  0.643574  0.648810  240.0000
weighted avg   0.724345  0.712500  0.711696  240.0000

Confusion matrix (rows=true, columns=predicted):
      none  low  high
none    15   26     2
low     17   72     4
high     1   19    84

Per-book macro F1:
A farewell to arms                            n= 40 F1=0.6799
Dubliners                                     n= 40 F1=0.6298
Middlem


TEST EVALUATION
accuracy        0.6042 [0.5125, 0.6876] valid=1000/1000
macro_f1        0.5009 [0.4028, 0.5805] valid=1000/1000
weighted_f1     0.5799 [0.4848, 0.6711] valid=1000/1000
micro_f1        0.6042 [0.5125, 0.6876] valid=1000/1000
spearman        0.5476 [0.3818, 0.6749] valid=1000/1000

Per-class precision / recall / F1:
              precision    recall  f1-score     support
none           0.357143  0.116279  0.175439   43.000000
low            0.512195  0.677419  0.583333   93.000000
high           0.747573  0.740385  0.743961  104.000000
accuracy       0.604167  0.604167  0.604167    0.604167
macro avg      0.538970  0.511361  0.500911  240.000000
weighted avg   0.586412  0.604167  0.579858  240.000000

Confusion matrix (rows=true, columns=predicted):
      none  low  high
none     5   34     4
low      8   63    22
high     1   26    77

Per-book macro F1:
A farewell to arms                            n= 40 F1=0.5870
Dubliners                                     n= 40 F1=

# Final comparison summary

In [ ]:
SUMMARY_METRICS = [
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "micro_f1",
    "spearman",
    "f1_none",
    "f1_low",
    "f1_high",
]

summary_rows = []
per_book_rows = []

for model_name, results in test_results.items():
    print(f"\n{'=' * 70}\n{model_name}\n{'=' * 70}")
    config = best_per_model[model_name]["config"]
    dev_macro_f1 = best_per_model[model_name]["dev_results"]["macro_f1"]

    print("Best configuration:", config)
    print("DEV macro F1:", f"{dev_macro_f1:.4f}")

    row = {
        "model": model_name,
        "best_learning_rate": config["learning_rate"],
        "best_batch_size": config["batch_size"],
        "best_warmup_ratio": config["warmup_ratio"],
        "dev_macro_f1": dev_macro_f1,
    }

    for metric in SUMMARY_METRICS:
        point = results[metric]
        lower = results[f"{metric}_lo"]
        upper = results[f"{metric}_hi"]
        valid = results[f"{metric}_bootstrap_valid"]

        print(
            f"{metric:<15} {point:.4f} [{lower:.4f}, {upper:.4f}] "
            f"valid={valid}/{N_BOOTSTRAP}"
        )

        row[metric] = point
        row[f"{metric}_ci_lower"] = lower
        row[f"{metric}_ci_upper"] = upper
        row[f"{metric}_bootstrap_valid"] = valid

    summary_rows.append(row)

    print("\nPer-book TEST macro F1:")
    for book, score in results["per_book"].items():
        print(f"{book:<45} {score:.4f}")
        per_book_rows.append(
            {
                "model": model_name,
                "book": book,
                "test_macro_f1": score,
            }
        )

summary_df = pd.DataFrame(summary_rows)
per_book_df = pd.DataFrame(per_book_rows)

summary_dir = OUTPUT_ROOT / "summary"
summary_dir.mkdir(parents=True, exist_ok=True)

summary_path = summary_dir / "all_encoder_models_test_summary.csv"
per_book_path = summary_dir / "all_encoder_models_per_book_f1.csv"

summary_df.to_csv(summary_path, index=False)
per_book_df.to_csv(per_book_path, index=False)

print(f"\nSaved final summary → {summary_path}")
print(f"Saved per-book summary → {per_book_path}")

display(summary_df)



bert-base-uncased
Best configuration: {'learning_rate': 3e-05, 'batch_size': 16, 'warmup_ratio': 0.1}
DEV macro F1: 0.5820
accuracy        0.6583 [0.5792, 0.7417] valid=1000/1000
macro_f1        0.5676 [0.4791, 0.6424] valid=1000/1000
weighted_f1     0.6374 [0.5563, 0.7203] valid=1000/1000
micro_f1        0.6583 [0.5792, 0.7417] valid=1000/1000
spearman        0.5730 [0.3946, 0.7061] valid=1000/1000
f1_none         0.2759 [0.1154, 0.4187] valid=1000/1000
f1_low          0.6606 [0.5454, 0.7359] valid=1000/1000
f1_high         0.7662 [0.6044, 0.8696] valid=1000/1000

Per-book TEST macro F1:
A farewell to arms                            0.5922
Dubliners                                     0.6528
Middlemarch                                   0.6891
My Ántonia                                    0.3913
The Picture of Dorian Gray                    0.4313
The murder of Roger Ackroyd                   0.5378

roberta-base
Best configuration: {'learning_rate': 3e-05, 'batch_size': 16, 'warmup_

,model,best_learning_rate,best_batch_size,best_warmup_ratio,dev_macro_f1,accuracy,accuracy_ci_lower,accuracy_ci_upper,accuracy_bootstrap_valid,macro_f1,...,f1_none_ci_upper,f1_none_bootstrap_valid,f1_low,f1_low_ci_lower,f1_low_ci_upper,f1_low_bootstrap_valid,f1_high,f1_high_ci_lower,f1_high_ci_upper,f1_high_bootstrap_valid
0,bert-base-uncased,0.00003,16,0.10,0.582038,0.658333,0.579167,0.741667,1000,0.567555,...,0.418716,1000,0.660633,0.545432,0.735943,1000,0.766169,0.604374,0.869565,1000
1,roberta-base,0.00003,16,0.06,0.684543,0.712500,0.666667,0.775104,1000,0.648810,...,0.493151,1000,0.685714,0.640394,0.722892,1000,0.865979,0.810526,0.913462,1000
2,distilbert-base-uncased,0.00003,16,0.06,0.629011,0.604167,0.512500,0.687604,1000,0.500911,...,0.293353,1000,0.583333,0.490352,0.654899,1000,0.743961,0.613241,0.850252,1000
3,answerdotai/ModernBERT-base,0.00001,16,0.10,0.599838,0.566667,0.508333,0.650104,1000,0.457407,...,0.151573,1000,0.582278,0.502513,0.644195,1000,0.680851,0.533223,0.805687,1000
